###

### Why was the attention mechanism invented, and what problem does it solve?

### What is residual connection?
### How attention architecture improves language models?



## Attention


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

B = 4  # Batch size
T = 8  # Time steps (sequence length)
C = 2  # Channels (features per token)
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)

In [ ]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [ ]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [ ]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)

False

### using softmax

In [ ]:
tril = torch.tril(torch.ones(T, T))
print(f"tril:{tril}")
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
print(f"masked:{wei}")
wei = F.softmax(wei, dim=-1)
wei

tril:tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])
masked:tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [ ]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)


trail:
 tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])
wei:
 tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])


False

In [ ]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
print(a)
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [ ]:
# consider the following toy example:

torch.manual_seed(1337)
B = 4  # Batch size
T = 8  # Time steps (sequence length)
C = 2  # Channels (features per token)
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
x

tensor([[[ 0.1808, -0.0700],
         [-0.3596, -0.9152],
         [ 0.6258,  0.0255],
         [ 0.9545,  0.0643],
         [ 0.3612,  1.1679],
         [-1.3499, -0.5102],
         [ 0.2360, -0.2398],
         [-0.9211,  1.5433]],

        [[ 1.3488, -0.1396],
         [ 0.2858,  0.9651],
         [-2.0371,  0.4931],
         [ 1.4870,  0.5910],
         [ 0.1260, -1.5627],
         [-1.1601, -0.3348],
         [ 0.4478, -0.8016],
         [ 1.5236,  2.5086]],

        [[-0.6631, -0.2513],
         [ 1.0101,  0.1215],
         [ 0.1584,  1.1340],
         [-1.1539, -0.2984],
         [-0.5075, -0.9239],
         [ 0.5467, -1.4948],
         [-1.2057,  0.5718],
         [-0.5974, -0.6937]],

        [[ 1.6455, -0.8030],
         [ 1.3514, -0.2759],
         [-1.5108,  2.1048],
         [ 2.7630, -1.7465],
         [ 1.4516, -1.5103],
         [ 0.8212, -0.2115],
         [ 0.7789,  1.5333],
         [ 1.6097, -0.4032]]])

In [ ]:
"""
Encoder self-attention + Decoder masked self-attention + Decoder cross-attention
Toy numpy walkthrough for: "I love playing football"

Single head, d_model = d_k = d_v = 4, no batching, weights randomly initialized
just to make the mechanics concrete and inspectable.
"""

import numpy as np

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(42)


def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)


def self_attention(X, Wq, Wk, Wv, mask=None):
    """
    X:    (seq_len, d_model) input representations
    Wq/Wk/Wv: (d_model, d_model) projection matrices
    mask: optional (seq_len, seq_len) boolean array, True = block that position
    Returns: (output, attn_weights)
    """
    d_k = Wq.shape[-1]
    Q = X @ Wq
    K = X @ Wk
    V = X @ Wv

    scores = Q @ K.T / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, -np.inf, scores)

    attn_weights = softmax(scores)
    output = attn_weights @ V
    return output, attn_weights


def cross_attention(X_decoder, X_encoder, Wq, Wk, Wv):
    """
    Query comes from the decoder side, Key/Value come from the encoder side.
    No causal mask: decoder can attend to the full source sequence.
    """
    d_k = Wq.shape[-1]
    Q = X_decoder @ Wq
    K = X_encoder @ Wk
    V = X_encoder @ Wv

    scores = Q @ K.T / np.sqrt(d_k)
    attn_weights = softmax(scores)
    output = attn_weights @ V
    return output, attn_weights


def causal_mask(seq_len):
    """Upper-triangular mask (excluding diagonal): position i can't see j > i."""
    return np.triu(np.ones((seq_len, seq_len)), k=1).astype(bool)


if __name__ == "__main__":
    d_model = 4

    # ---- Encoder side: "I love playing football" ----
    tokens_enc = ["I", "love", "playing", "football"]
    X_enc = np.array([
        [1.0, 0.0, 1.0, 0.0],
        [0.0, 1.0, 0.0, 1.0],
        [1.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 1.0],
    ])

    Wq_enc = rng.normal(scale=0.5, size=(d_model, d_model))
    Wk_enc = rng.normal(scale=0.5, size=(d_model, d_model))
    Wv_enc = rng.normal(scale=0.5, size=(d_model, d_model))

    encoder_output, enc_attn = self_attention(X_enc, Wq_enc, Wk_enc, Wv_enc)

    print("=== Encoder self-attention ===")
    print("attention weights:\n", enc_attn)
    print("encoder output:\n", encoder_output)

    # ---- Decoder side: "<start> I love playing" (teacher forcing) ----
    tokens_dec = ["<start>", "I", "love", "playing"]
    X_dec = np.array([
        [0.5, 0.5, 0.5, 0.5],
        [1.0, 0.0, 1.0, 0.0],
        [0.0, 1.0, 0.0, 1.0],
        [1.0, 1.0, 0.0, 0.0],
    ])

    Wq_dec = rng.normal(scale=0.5, size=(d_model, d_model))
    Wk_dec = rng.normal(scale=0.5, size=(d_model, d_model))
    Wv_dec = rng.normal(scale=0.5, size=(d_model, d_model))

    print(f"Wk_dec:{Wq_dec}")

    mask = causal_mask(len(tokens_dec))
    decoder_self_out, dec_self_attn = self_attention(
        X_dec, Wq_dec, Wk_dec, Wv_dec, mask=mask
    )

    print("\n=== Decoder masked self-attention ===")
    print("causal mask (True = blocked):\n", mask)
    print("attention weights:\n", dec_self_attn)
    print("decoder self-attn output:\n", decoder_self_out)

    # ---- Decoder cross-attention: Q from decoder, K/V from encoder ----
    Wq_cross = rng.normal(scale=0.5, size=(d_model, d_model))
    Wk_cross = rng.normal(scale=0.5, size=(d_model, d_model))
    Wv_cross = rng.normal(scale=0.5, size=(d_model, d_model))

    decoder_final, cross_attn = cross_attention(
        decoder_self_out, encoder_output, Wq_cross, Wk_cross, Wv_cross
    )

    print("\n=== Decoder cross-attention ===")
    print("attention weights (rows=decoder pos, cols=encoder pos):\n", cross_attn)
    print("final decoder output (pre-FFN):\n", decoder_final)

=== Encoder self-attention ===
attention weights:
 [[0.253 0.245 0.284 0.218]
 [0.228 0.273 0.246 0.253]
 [0.276 0.221 0.3   0.204]
 [0.205 0.299 0.229 0.267]]
encoder output:
 [[ 0.035 -0.262 -0.008  0.581]
 [ 0.058 -0.223  0.001  0.551]
 [ 0.026 -0.276 -0.012  0.597]
 [ 0.067 -0.208  0.006  0.535]]
Wk_dec:[[ 0.339  0.034  0.145  0.316]
 [-0.729 -0.16  -0.235 -0.319]
 [-0.138  0.747 -0.433  0.484]
 [-0.841 -0.167  0.081  0.293]]

=== Decoder masked self-attention ===
causal mask (True = blocked):
 [[False  True  True  True]
 [False False  True  True]
 [False False False  True]
 [False False False False]]
attention weights:
 [[1.    0.    0.    0.   ]
 [0.434 0.566 0.    0.   ]
 [0.333 0.353 0.314 0.   ]
 [0.257 0.258 0.256 0.229]]
decoder self-attn output:
 [[-0.17   0.048 -0.464 -0.479]
 [ 0.191 -0.04  -0.116 -0.344]
 [-0.146  0.042 -0.44  -0.47 ]
 [-0.215  0.017 -0.451 -0.411]]

=== Decoder cross-attention ===
attention weights (rows=decoder pos, cols=encoder pos):
 [[0.251 0.249 0.

In [1]:
import numpy as np

print("=== COMPLETE SELF-ATTENTION WITH Wq, Wk, Wv ===")
print("Sentence: 'I love playing' -> 3 tokens")
print("="*60 + "\n")

# ============= SETUP =============
B, T, d_model, h = 1, 3, 4, 2  # 3 tokens, 4 dims, 2 heads
d_k = d_model // h  # 2

# Input X (after embedding + positional encoding)
# Shape: [B=1, T=3, d_model=4]
X = np.array([[[1, 2, 3, 4],    # Token 1: "I"
               [5, 6, 7, 8],    # Token 2: "love"
               [9, 10, 11, 12]]]) # Token 3: "playing"

print("INPUT X (from embedding + positional encoding):")
print(f"Shape: {X.shape} [B, T, d_model]")
print("X Matrix (each row = one token):")
print(X[0])
print("\n" + "="*60 + "\n")

# ============= STEP 1: CREATE WEIGHT MATRICES =============
print("=== STEP 1: CREATE Wq, Wk, Wv ===")

# These are LEARNED parameters (randomly initialized here)
Wq = np.array([[0.1, 0.2, 0.3, 0.4],
               [0.5, 0.6, 0.7, 0.8],
               [0.9, 1.0, 1.1, 1.2],
               [1.3, 1.4, 1.5, 1.6]])

Wk = np.array([[0.2, 0.4, 0.6, 0.8],
               [1.0, 1.2, 1.4, 1.6],
               [1.8, 2.0, 2.2, 2.4],
               [2.6, 2.8, 3.0, 3.2]])

Wv = np.array([[0.3, 0.6, 0.9, 1.2],
               [1.5, 1.8, 2.1, 2.4],
               [2.7, 3.0, 3.3, 3.6],
               [3.9, 4.2, 4.5, 4.8]])

print(f"Wq Shape: {Wq.shape} [d_model, d_model]")
print("Wq (Query weights):")
print(Wq)
print("\nWk (Key weights):")
print(Wk)
print("\nWv (Value weights):")
print(Wv)
print("\n" + "="*60 + "\n")

# ============= STEP 2: COMPUTE Q, K, V =============
print("=== STEP 2: COMPUTE Q, K, V ===")
print("Q = X @ Wq, K = X @ Wk, V = X @ Wv")

# Matrix multiplication: [B, T, d_model] @ [d_model, d_model] = [B, T, d_model]
Q = X @ Wq
K = X @ Wk
V = X @ Wv

print(f"Q Shape: {Q.shape} [B, T, d_model]")
print("Q Matrix (Queries for each token):")
print(Q[0])
print("\nK Matrix (Keys for each token):")
print(K[0])
print("\nV Matrix (Values for each token):")
print(V[0])
print("\n" + "="*60 + "\n")

# ============= STEP 3: RESHAPE (View) =============
print("=== STEP 3: RESHAPE (Split into Heads) ===")
print("[B, T, d_model] -> [B, T, h, d_k]")

Q_view = Q.reshape(B, T, h, d_k)
K_view = K.reshape(B, T, h, d_k)
V_view = V.reshape(B, T, h, d_k)

print(f"Reshaped Q Shape: {Q_view.shape} [B, T, h, d_k]")
print("Notice how the 4 dimensions are split into 2 pairs (2 heads):")
print("\nQ View (all tokens, separated by head):")
print("Token 1 ('I'):")
print(f"  Head 0: {Q_view[0, 0, 0]}  (first 2 dims)")
print(f"  Head 1: {Q_view[0, 0, 1]}  (last 2 dims)")
print("Token 2 ('love'):")
print(f"  Head 0: {Q_view[0, 1, 0]}")
print(f"  Head 1: {Q_view[0, 1, 1]}")
print("Token 3 ('playing'):")
print(f"  Head 0: {Q_view[0, 2, 0]}")
print(f"  Head 1: {Q_view[0, 2, 1]}")
print("\n" + "="*60 + "\n")

# ============= STEP 4: TRANSPOSE =============
print("=== STEP 4: TRANSPOSE ===")
print("[B, T, h, d_k] -> [B, h, T, d_k]")

Q_trans = np.transpose(Q_view, (0, 2, 1, 3))
K_trans = np.transpose(K_view, (0, 2, 1, 3))
V_trans = np.transpose(V_view, (0, 2, 1, 3))

print(f"Transposed Q Shape: {Q_trans.shape} [B, h, T, d_k]")
print("Now Head 0 and Head 1 are completely separated:")
print("\n--- Q Head 0 (all tokens, first 2 dims) ---")
print(Q_trans[0, 0])  # Shape [T=3, d_k=2]
print("\n--- Q Head 1 (all tokens, last 2 dims) ---")
print(Q_trans[0, 1])  # Shape [T=3, d_k=2]

print("\n--- K Head 0 ---")
print(K_trans[0, 0])
print("\n--- K Head 1 ---")
print(K_trans[0, 1])

print("\n--- V Head 0 ---")
print(V_trans[0, 0])
print("\n--- V Head 1 ---")
print(V_trans[0, 1])
print("\n" + "="*60 + "\n")

# ============= STEP 5: ATTENTION MATH =============
print("=== STEP 5: ATTENTION MATH ===")
print("scores = (Q @ K^T) / sqrt(d_k)")
print(f"d_k = {d_k}, sqrt(d_k) = {np.sqrt(d_k):.2f}")

# [B, h, T, d_k] @ [B, h, d_k, T] = [B, h, T, T]
scores = Q_trans @ np.transpose(K_trans, (0, 1, 3, 2))
scores = scores / np.sqrt(d_k)  # Scale

print(f"Scores Shape: {scores.shape} [B, h, T, T]")
print("\n--- Scores Head 0 (3x3 matrix) ---")
print(scores[0, 0])
print("\n--- Scores Head 1 (3x3 matrix) ---")
print(scores[0, 1])

# Softmax
print("\nApplying Softmax...")
scores_exp = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
attn_weights = scores_exp / np.sum(scores_exp, axis=-1, keepdims=True)

print(f"Attention Weights Shape: {attn_weights.shape} [B, h, T, T]")
print("\n--- Attention Weights Head 0 (rows sum to 1) ---")
print(attn_weights[0, 0])
print("Row 0 sum:", attn_weights[0, 0, 0].sum())
print("\n--- Attention Weights Head 1 ---")
print(attn_weights[0, 1])

# Apply attention to values: attn @ V
head_out = attn_weights @ V_trans

print(f"\nHead Output Shape: {head_out.shape} [B, h, T, d_k]")
print("\n--- Head 0 Output (each token's new representation) ---")
print(head_out[0, 0])
print("\n--- Head 1 Output ---")
print(head_out[0, 1])
print("\n" + "="*60 + "\n")

# ============= STEP 6: CONCATENATE =============
print("=== STEP 6: CONCATENATE HEADS ===")
print("[B, h, T, d_k] -> [B, T, h, d_k] -> [B, T, d_model]")

# Reverse transpose: [B, h, T, d_k] -> [B, T, h, d_k]
head_out_transposed = np.transpose(head_out, (0, 2, 1, 3))

# Concatenate heads: [B, T, h, d_k] -> [B, T, d_model]
final_concat = head_out_transposed.reshape(B, T, d_model)

print(f"Final Concatenated Shape: {final_concat.shape} [B, T, d_model]")
print("The pairs from Head 0 and Head 1 are glued back together:")
print("\n--- Final Output (each token with full 4-dim representation) ---")
print("Token 1 ('I'):     ", final_concat[0, 0])
print("Token 2 ('love'):   ", final_concat[0, 1])
print("Token 3 ('playing'):", final_concat[0, 2])
print("\n" + "="*60 + "\n")

# ============= STEP 7: OUTPUT PROJECTION =============
print("=== STEP 7: OUTPUT PROJECTION ===")
print("MHA_output = final_concat @ Wo")

# Wo is another learned weight matrix
Wo = np.array([[0.1, 0.2, 0.3, 0.4],
               [0.5, 0.6, 0.7, 0.8],
               [0.9, 1.0, 1.1, 1.2],
               [1.3, 1.4, 1.5, 1.6]])

print(f"Wo Shape: {Wo.shape} [d_model, d_model]")
mha_output = final_concat @ Wo

print(f"MHA Output Shape: {mha_output.shape} [B, T, d_model]")
print("\n--- MHA Output (after final projection) ---")
print("Token 1 ('I'):     ", mha_output[0, 0])
print("Token 2 ('love'):   ", mha_output[0, 1])
print("Token 3 ('playing'):", mha_output[0, 2])
print("\n" + "="*60 + "\n")

# ============= VERIFICATION =============
print("=== VERIFICATION: WHAT EACH TOKEN LEARNED ===")
print("\nOriginal Input:")
print("Token 1 ('I'):     ", X[0, 0])
print("Token 2 ('love'):   ", X[0, 1])
print("Token 3 ('playing'):", X[0, 2])

print("\nAfter Self-Attention (each token now contains info from ALL tokens):")
print("Token 1 ('I'):     ", mha_output[0, 0])
print("Token 2 ('love'):   ", mha_output[0, 1])
print("Token 3 ('playing'):", mha_output[0, 2])

print("\n✅ Each token's representation has been updated with context!")

=== COMPLETE SELF-ATTENTION WITH Wq, Wk, Wv ===
Sentence: 'I love playing' -> 3 tokens

INPUT X (from embedding + positional encoding):
Shape: (1, 3, 4) [B, T, d_model]
X Matrix (each row = one token):
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]


=== STEP 1: CREATE Wq, Wk, Wv ===
Wq Shape: (4, 4) [d_model, d_model]
Wq (Query weights):
[[0.1 0.2 0.3 0.4]
 [0.5 0.6 0.7 0.8]
 [0.9 1.  1.1 1.2]
 [1.3 1.4 1.5 1.6]]

Wk (Key weights):
[[0.2 0.4 0.6 0.8]
 [1.  1.2 1.4 1.6]
 [1.8 2.  2.2 2.4]
 [2.6 2.8 3.  3.2]]

Wv (Value weights):
[[0.3 0.6 0.9 1.2]
 [1.5 1.8 2.1 2.4]
 [2.7 3.  3.3 3.6]
 [3.9 4.2 4.5 4.8]]


=== STEP 2: COMPUTE Q, K, V ===
Q = X @ Wq, K = X @ Wk, V = X @ Wv
Q Shape: (1, 3, 4) [B, T, d_model]
Q Matrix (Queries for each token):
[[ 9.  10.  11.  12. ]
 [20.2 22.8 25.4 28. ]
 [31.4 35.6 39.8 44. ]]

K Matrix (Keys for each token):
[[18.  20.  22.  24. ]
 [40.4 45.6 50.8 56. ]
 [62.8 71.2 79.6 88. ]]

V Matrix (Values for each token):
[[ 27.   30.   33.   36. ]
 [ 60.6  68.4  

In [34]:
import numpy as np

token = np.arange(1,13).reshape(1,3, 4)
w_q= w_k=w_v= np.arange(1,17).reshape(1,4, 4)
token, w_q, w_k,w_v

(array([[[ 1,  2,  3,  4],
         [ 5,  6,  7,  8],
         [ 9, 10, 11, 12]]]),
 array([[[ 1,  2,  3,  4],
         [ 5,  6,  7,  8],
         [ 9, 10, 11, 12],
         [13, 14, 15, 16]]]),
 array([[[ 1,  2,  3,  4],
         [ 5,  6,  7,  8],
         [ 9, 10, 11, 12],
         [13, 14, 15, 16]]]),
 array([[[ 1,  2,  3,  4],
         [ 5,  6,  7,  8],
         [ 9, 10, 11, 12],
         [13, 14, 15, 16]]]))

In [35]:
Q=token@w_q
K = token@w_k
V = token@w_v
Q, K, V

(array([[[ 90, 100, 110, 120],
         [202, 228, 254, 280],
         [314, 356, 398, 440]]]),
 array([[[ 90, 100, 110, 120],
         [202, 228, 254, 280],
         [314, 356, 398, 440]]]),
 array([[[ 90, 100, 110, 120],
         [202, 228, 254, 280],
         [314, 356, 398, 440]]]))

In [16]:
dk=4/2

In [37]:
q_view = Q.reshape(1, 3, 2, 2)
k_view = K.reshape(1, 3, 2, 2)
v_view = V.reshape(1, 3, 2, 2)
q_view, k_view, v_view


(array([[[[ 90, 100],
          [110, 120]],
 
         [[202, 228],
          [254, 280]],
 
         [[314, 356],
          [398, 440]]]]),
 array([[[[ 90, 100],
          [110, 120]],
 
         [[202, 228],
          [254, 280]],
 
         [[314, 356],
          [398, 440]]]]),
 array([[[[ 90, 100],
          [110, 120]],
 
         [[202, 228],
          [254, 280]],
 
         [[314, 356],
          [398, 440]]]]))

In [38]:
q_trans = np.transpose(q_view, (0, 2, 1, 3))
k_trans = np.transpose(k_view, (0, 2, 1, 3))
v_trans = np.transpose(v_view, (0, 2, 1, 3))
q_trans, k_trans, v_trans

(array([[[[ 90, 100],
          [202, 228],
          [314, 356]],
 
         [[110, 120],
          [254, 280],
          [398, 440]]]]),
 array([[[[ 90, 100],
          [202, 228],
          [314, 356]],
 
         [[110, 120],
          [254, 280],
          [398, 440]]]]),
 array([[[[ 90, 100],
          [202, 228],
          [314, 356]],
 
         [[110, 120],
          [254, 280],
          [398, 440]]]]))

In [40]:
k_trans2 = np.transpose(k_trans, (0, 1, 3, 2))
k_trans2

array([[[[ 90, 202, 314],
         [100, 228, 356]],

        [[110, 254, 398],
         [120, 280, 440]]]])

In [44]:
socre_0 = q_trans @ k_trans2
head_out = socre_0 @ V_trans
head_out,head_out.shape

(array([[[[ 8987700.        , 10166280.        ],
          [20350356.        , 23018952.        ],
          [31713012.        , 35871624.        ]],
 
         [[17095500.        , 18871920.        ],
          [39701484.        , 43826928.        ],
          [62307468.00000001, 68781936.        ]]]]),
 (1, 2, 3, 2))

In [46]:
head_out_transposed = np.transpose(head_out, (0, 2, 1, 3))
head_out_transposed, head_out_transposed.shape

(array([[[[ 8987700.        , 10166280.        ],
          [17095500.        , 18871920.        ]],
 
         [[20350356.        , 23018952.        ],
          [39701484.        , 43826928.        ]],
 
         [[31713012.        , 35871624.        ],
          [62307468.00000001, 68781936.        ]]]]),
 (1, 3, 2, 2))

In [47]:
final_concat = head_out_transposed.reshape(B, T, d_model)
final_concat, final_concat.shape

(array([[[ 8987700.        , 10166280.        , 17095500.        ,
          18871920.        ],
         [20350356.        , 23018952.        , 39701484.        ,
          43826928.        ],
         [31713012.        , 35871624.        , 62307468.00000001,
          68781936.        ]]]),
 (1, 3, 4))

In [32]:
import numpy as np


# ============= SETUP =============
B, T, d_model, h = 1, 3, 4, 2  # 3 tokens, 4 dims, 2 heads
d_k = d_model // h  # 2

def initialize_hyper_parameter(isPrinting=False):
  # Input X (after embedding + positional encoding)
  # Shape: [B=1, T=3, d_model=4]
  X = np.array([[[1, 2, 3, 4],    # Token 1: "I"
                [5, 6, 7, 8],    # Token 2: "love"
                [9, 10, 11, 12]]]) # Token 3: "playing"

  if isPrinting:
    print("INPUT X (from embedding + positional encoding):")
    print(f"Shape: {X.shape} [B, T, d_model]")
    print("X Matrix (each row = one token):")
    print(X[0])
    print("\n" + "="*60 + "\n")

  # ============= STEP 1: CREATE WEIGHT MATRICES =============
  print("=== STEP 1: CREATE Wq, Wk, Wv ===")

  # These are LEARNED parameters (randomly initialized here)
  Wq = np.array([[0.1, 0.2, 0.3, 0.4],
                [0.5, 0.6, 0.7, 0.8],
                [0.9, 1.0, 1.1, 1.2],
                [1.3, 1.4, 1.5, 1.6]])

  Wk = np.array([[0.2, 0.4, 0.6, 0.8],
                [1.0, 1.2, 1.4, 1.6],
                [1.8, 2.0, 2.2, 2.4],
                [2.6, 2.8, 3.0, 3.2]])

  Wv = np.array([[0.3, 0.6, 0.9, 1.2],
                [1.5, 1.8, 2.1, 2.4],
                [2.7, 3.0, 3.3, 3.6],
                [3.9, 4.2, 4.5, 4.8]])

  if isPrinting:
    print(f"Wq Shape: {Wq.shape} [d_model, d_model]")
    print("Wq (Query weights):")
    print(Wq)
    print("\nWk (Key weights):")
    print(Wk)
    print("\nWv (Value weights):")
    print(Wv)
    print("\n" + "="*60 + "\n")
  return X, Wq, Wk, Wv

X, Wq, Wk, Wv= initialize_hyper_parameter()



# ============= STEP 2: COMPUTE Q, K, V =============
print("=== STEP 2: COMPUTE Q, K, V ===")
print("Q = X @ Wq, K = X @ Wk, V = X @ Wv")

# Matrix multiplication: [B, T, d_model] @ [d_model, d_model] = [B, T, d_model]
Q = X @ Wq
K = X @ Wk
V = X @ Wv

print(f"Q Shape: {Q.shape} [B, T, d_model]")
print("Q Matrix (Queries for each token):")
print(Q[0])
print("\nK Matrix (Keys for each token):")
print(K[0])
print("\nV Matrix (Values for each token):")
print(V[0])
print("\n" + "="*60 + "\n")

# ============= STEP 3: RESHAPE (View) =============
print("=== STEP 3: RESHAPE (Split into Heads) ===")
print("[B, T, d_model] -> [B, T, h, d_k]")

Q_view = Q.reshape(B, T, h, d_k)
K_view = K.reshape(B, T, h, d_k)
V_view = V.reshape(B, T, h, d_k)

print(f"Reshaped Q Shape: {Q_view.shape} [B, T, h, d_k]")
print("Notice how the 4 dimensions are split into 2 pairs (2 heads):")
print("\nQ View (all tokens, separated by head):")
print("Token 1 ('I'):")
print(f"  Head 0: {Q_view[0, 0, 0]}  (first 2 dims)")
print(f"  Head 1: {Q_view[0, 0, 1]}  (last 2 dims)")
print("Token 2 ('love'):")
print(f"  Head 0: {Q_view[0, 1, 0]}")
print(f"  Head 1: {Q_view[0, 1, 1]}")
print("Token 3 ('playing'):")
print(f"  Head 0: {Q_view[0, 2, 0]}")
print(f"  Head 1: {Q_view[0, 2, 1]}")
print("\n" + "="*60 + "\n")

# ============= STEP 4: TRANSPOSE =============
print("=== STEP 4: TRANSPOSE ===")
print("[B, T, h, d_k] -> [B, h, T, d_k]")

Q_trans = np.transpose(Q_view, (0, 2, 1, 3))
K_trans = np.transpose(K_view, (0, 2, 1, 3))
V_trans = np.transpose(V_view, (0, 2, 1, 3))

print(f"Transposed Q Shape: {Q_trans.shape} [B, h, T, d_k]")
print("Now Head 0 and Head 1 are completely separated:")
print("\n--- Q Head 0 (all tokens, first 2 dims) ---")
print(Q_trans[0, 0])  # Shape [T=3, d_k=2]
print("\n--- Q Head 1 (all tokens, last 2 dims) ---")
print(Q_trans[0, 1])  # Shape [T=3, d_k=2]

print("\n--- K Head 0 ---")
print(K_trans[0, 0])
print("\n--- K Head 1 ---")
print(K_trans[0, 1])

print("\n--- V Head 0 ---")
print(V_trans[0, 0])
print("\n--- V Head 1 ---")
print(V_trans[0, 1])
print("\n" + "="*60 + "\n")

# ============= STEP 5: ATTENTION MATH =============
print("=== STEP 5: ATTENTION MATH ===")
print("scores = (Q @ K^T) / sqrt(d_k)")
print(f"d_k = {d_k}, sqrt(d_k) = {np.sqrt(d_k):.2f}")

# [B, h, T, d_k] @ [B, h, d_k, T] = [B, h, T, T]
scores = Q_trans @ np.transpose(K_trans, (0, 1, 3, 2))
scores = scores / np.sqrt(d_k)  # Scale

print(f"Scores Shape: {scores.shape} [B, h, T, T]")
print("\n--- Scores Head 0 (3x3 matrix) ---")
print(scores[0, 0])
print("\n--- Scores Head 1 (3x3 matrix) ---")
print(scores[0, 1])

# Softmax
print("\nApplying Softmax...")
scores_exp = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
attn_weights = scores_exp / np.sum(scores_exp, axis=-1, keepdims=True)

print(f"Attention Weights Shape: {attn_weights.shape} [B, h, T, T]")
print("\n--- Attention Weights Head 0 (rows sum to 1) ---")
print(attn_weights[0, 0])
print("Row 0 sum:", attn_weights[0, 0, 0].sum())
print("\n--- Attention Weights Head 1 ---")
print(attn_weights[0, 1])

# Apply attention to values: attn @ V
head_out = attn_weights @ V_trans

print(f"\nHead Output Shape: {head_out.shape} [B, h, T, d_k]")
print("\n--- Head 0 Output (each token's new representation) ---")
print(head_out[0, 0])
print("\n--- Head 1 Output ---")
print(head_out[0, 1])
print("\n" + "="*60 + "\n")

# ============= STEP 6: CONCATENATE =============
print("=== STEP 6: CONCATENATE HEADS ===")
print("[B, h, T, d_k] -> [B, T, h, d_k] -> [B, T, d_model]")

# Reverse transpose: [B, h, T, d_k] -> [B, T, h, d_k]
head_out_transposed = np.transpose(head_out, (0, 2, 1, 3))

# Concatenate heads: [B, T, h, d_k] -> [B, T, d_model]
final_concat = head_out_transposed.reshape(B, T, d_model)

print(f"Final Concatenated Shape: {final_concat.shape} [B, T, d_model]")
print("The pairs from Head 0 and Head 1 are glued back together:")
print("\n--- Final Output (each token with full 4-dim representation) ---")
print("Token 1 ('I'):     ", final_concat[0, 0])
print("Token 2 ('love'):   ", final_concat[0, 1])
print("Token 3 ('playing'):", final_concat[0, 2])
print("\n" + "="*60 + "\n")

# ============= STEP 7: OUTPUT PROJECTION =============
print("=== STEP 7: OUTPUT PROJECTION ===")
print("MHA_output = final_concat @ Wo")

# Wo is another learned weight matrix
Wo = np.array([[0.1, 0.2, 0.3, 0.4],
               [0.5, 0.6, 0.7, 0.8],
               [0.9, 1.0, 1.1, 1.2],
               [1.3, 1.4, 1.5, 1.6]])

print(f"Wo Shape: {Wo.shape} [d_model, d_model]")
mha_output = final_concat @ Wo

print(f"MHA Output Shape: {mha_output.shape} [B, T, d_model]")
print("\n--- MHA Output (after final projection) ---")
print("Token 1 ('I'):     ", mha_output[0, 0])
print("Token 2 ('love'):   ", mha_output[0, 1])
print("Token 3 ('playing'):", mha_output[0, 2])
print("\n" + "="*60 + "\n")

# ============= VERIFICATION =============
print("=== VERIFICATION: WHAT EACH TOKEN LEARNED ===")
print("\nOriginal Input:")
print("Token 1 ('I'):     ", X[0, 0])
print("Token 2 ('love'):   ", X[0, 1])
print("Token 3 ('playing'):", X[0, 2])

print("\nAfter Self-Attention (each token now contains info from ALL tokens):")
print("Token 1 ('I'):     ", mha_output[0, 0])
print("Token 2 ('love'):   ", mha_output[0, 1])
print("Token 3 ('playing'):", mha_output[0, 2])

print("\n✅ Each token's representation has been updated with context!")

=== STEP 1: CREATE Wq, Wk, Wv ===
=== STEP 2: COMPUTE Q, K, V ===
Q = X @ Wq, K = X @ Wk, V = X @ Wv
Q Shape: (1, 3, 4) [B, T, d_model]
Q Matrix (Queries for each token):
[[ 9.  10.  11.  12. ]
 [20.2 22.8 25.4 28. ]
 [31.4 35.6 39.8 44. ]]

K Matrix (Keys for each token):
[[18.  20.  22.  24. ]
 [40.4 45.6 50.8 56. ]
 [62.8 71.2 79.6 88. ]]

V Matrix (Values for each token):
[[ 27.   30.   33.   36. ]
 [ 60.6  68.4  76.2  84. ]
 [ 94.2 106.8 119.4 132. ]]


=== STEP 3: RESHAPE (Split into Heads) ===
[B, T, d_model] -> [B, T, h, d_k]
Reshaped Q Shape: (1, 3, 2, 2) [B, T, h, d_k]
Notice how the 4 dimensions are split into 2 pairs (2 heads):

Q View (all tokens, separated by head):
Token 1 ('I'):
  Head 0: [ 9. 10.]  (first 2 dims)
  Head 1: [11. 12.]  (last 2 dims)
Token 2 ('love'):
  Head 0: [20.2 22.8]
  Head 1: [25.4 28. ]
Token 3 ('playing'):
  Head 0: [31.4 35.6]
  Head 1: [39.8 44. ]


=== STEP 4: TRANSPOSE ===
[B, T, h, d_k] -> [B, h, T, d_k]
Transposed Q Shape: (1, 2, 3, 2) [B, 